# Game 8 - The Cross-Modal Truth Arena

**Team:** Ded_Sec

This completed notebook documents the final multimodal system and loads the
generated validation and public-test artifacts. The image expert is the
optimized 160-pixel scratch CNN from Game 6. The text expert is the local
MiniLM encoder plus logistic classifier and quality features from Game 6.

The relation layer measures canonical image-text pairing, whether both
components are known, model agreement, and the probability gap between the
two modalities. It compares image-only, text-only, weighted fusion,
relation-gated fusion, logistic stacking, gradient-boosted stacking, and the
average of the two stacking models.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
comparison = pd.read_csv(ROOT / "game8_model_comparison.csv")
evidence = pd.read_csv(ROOT / "game8_evidence_analysis.csv")
public = pd.read_csv(ROOT / "game8_public_predictions.csv")
comparison.sort_values("macro_f1", ascending=False)

,system,modality,accuracy,macro_f1,precision,recall,relation_strategy,selected
5,relation_gradient_boosting,fusion,0.9300,0.929996,0.930099,0.9300,nonlinear probability and canonical relation i...,True
6,relation_stack_average,fusion,0.9284,0.928390,0.928631,0.9284,mean of the logistic and gradient-boosted stac...,False
3,canonical_relation_gate,fusion,0.9248,0.924719,0.926636,0.9248,known canonical mismatch forces fake,False
4,relation_logistic_stack,fusion,0.9244,0.924382,0.924814,0.9244,probabilities + confidence + canonical relatio...,False
2,weighted_probability_fusion,fusion,0.7496,0.744417,0.771632,0.7496,"weighted average, image_weight=0.500",False
0,image_only,image,0.7268,0.723600,0.737813,0.7268,none,False
1,text_only,text,0.6604,0.652412,0.676638,0.6604,none,False


## Selected system

`relation_gradient_boosting` was selected by validation macro-F1.

- Supplied benchmark accuracy: 0.9300
- Supplied benchmark macro-F1: 0.9300
- Image weight in the simple fusion baseline: 0.500

Canonical mismatches are strong evidence for `fake`. For matched pairs, the
fusion model decides how much to trust the image and text experts from their
probabilities, confidence, agreement, and interaction features.

This benchmark score uses all canonical manifests available from earlier games.
It is a competition benchmark result, not a guaranteed accuracy on unrelated
real-world data.


In [2]:
pd.crosstab(
    evidence["primary_evidence"],
    [evidence["correct"], evidence["reviewer_flag"]],
    margins=True,
)

correct             False       True        All
reviewer_flag       False True False True      
primary_evidence                               
image                  34   39   625  285   983
image_text_relation    27   11  1054  103  1195
text                   24   19   146   79   268
uncertain               0   21     0   33    54
All                    85   90  1825  500  2500

In [3]:
errors = evidence.loc[~evidence["correct"]]
errors[[
    "sample_id", "true_label", "predicted_label", "confidence",
    "primary_evidence", "reviewer_flag", "error_category"
]].head(30)

,sample_id,true_label,predicted_label,confidence,primary_evidence,reviewer_flag,error_category
17,g8_validation__000017,fake,real,0.837272,text,False,modality_disagreement_error
20,g8_validation__000020,fake,real,0.960716,text,False,both_experts_wrong
24,g8_validation__000024,real,fake,0.999457,image_text_relation,True,modality_disagreement_error
27,g8_validation__000027,fake,real,0.879414,image,False,modality_disagreement_error
33,g8_validation__000033,real,fake,0.897074,image,True,modality_disagreement_error
53,g8_validation__000053,real,fake,0.626333,image,True,modality_disagreement_error
54,g8_validation__000054,fake,real,0.964105,image,False,both_experts_wrong
71,g8_validation__000071,fake,real,0.876296,image_text_relation,True,modality_disagreement_error
76,g8_validation__000076,fake,real,0.833467,image_text_relation,False,fusion_overruled_correct_expert
83,g8_validation__000083,real,fake,0.999465,image,False,fusion_overruled_correct_expert


In [4]:
required = [
    "sample_id", "predicted_label", "confidence",
    "primary_evidence", "reviewer_flag",
]
assert public.columns.tolist() == required
assert public["primary_evidence"].isin(
    ["image", "text", "image_text_relation", "uncertain"]
).all()
assert public["confidence"].between(0, 1).all()
public.head(10)

,sample_id,predicted_label,confidence,primary_evidence,reviewer_flag
0,g8_public_test__000000,real,0.991604,text,False
1,g8_public_test__000001,real,0.998230,image,False
2,g8_public_test__000002,fake,0.998091,image_text_relation,False
3,g8_public_test__000003,fake,0.996838,image_text_relation,False
4,g8_public_test__000004,real,0.998313,image,False
5,g8_public_test__000005,real,0.998230,image,False
6,g8_public_test__000006,fake,0.995117,image_text_relation,False
7,g8_public_test__000007,fake,0.998466,image_text_relation,True
8,g8_public_test__000008,fake,0.995758,image_text_relation,False
9,g8_public_test__000009,real,0.998345,image,True


## Trust and review policy

Cases are flagged for review when confidence is below 0.68, when the image and
text experts strongly disagree, or when a detected relation mismatch is not
supported with at least 0.90 confidence. The evidence field is intentionally
limited to the four values required by the release contract.
